<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/notebooks/hybrid_reference_flow_and_density_ratios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Native reference flow and density ratios

This notebook is the YAML-driven replacement for the original Exercise 5 workflow. It uses only `hnsbi-toolkit`: the ratio trainer, diagnostics, ONNX export, workspace writer, JAX likelihood, and Minuit inference are all native.

> If a previous Colab run already loaded packages that the setup must replace, the setup cell restarts the kernel once. After Colab reconnects, run the setup cell again.

In [2]:
from importlib.metadata import PackageNotFoundError, version as installed_version
from pathlib import Path
import os, signal, subprocess, sys

def distribution_version(name):
    try:
        return installed_version(name)
    except PackageNotFoundError:
        return None

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    loaded_versions = {
        name: getattr(sys.modules.get(name), '__version__', None)
        for name in ('numpy', 'jax', 'jaxlib')
    }
    jax_was_loaded = any(
        name == 'jax' or name.startswith(('jax.', 'jaxlib', 'jax_plugins'))
        for name in sys.modules
    )
    ROOT = Path('/content/drive/MyDrive/hsbi-toolkit')
    REPO = ROOT / 'hnsbi-toolkit'
    ROOT.mkdir(parents=True, exist_ok=True)
    if not (REPO / '.git').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/rafaellopesdesa/hnsbi-toolkit.git', str(REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[lhc,flows]'], check=True)

    # Colab installs JAX's CUDA plugin separately from jaxlib. If an
    # earlier dependency resolution changed jaxlib, realign the plugin
    # before JAX discovers it; mixed PJRT versions fail at execution.
    jaxlib_version = distribution_version('jaxlib')
    plugin_extras = {
        'jax-cuda12-plugin': 'cuda12-local',
        'jax-cuda13-plugin': 'cuda13-local',
    }
    repaired_plugins = []
    for plugin, extra in plugin_extras.items():
        plugin_version = distribution_version(plugin)
        if plugin_version is not None and plugin_version != jaxlib_version:
            print(f'Aligning {plugin} {plugin_version} with jaxlib {jaxlib_version}.')
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                 f'jax[{extra}]=={jaxlib_version}'],
                check=True,
            )
            if distribution_version(plugin) != jaxlib_version:
                raise RuntimeError(f'Could not align {plugin} with jaxlib.')
            repaired_plugins.append(plugin)

    changed_loaded_packages = [
        name for name, loaded in loaded_versions.items()
        if loaded is not None and loaded != distribution_version(name)
    ]
    if changed_loaded_packages or (repaired_plugins and jax_was_loaded):
        reasons = changed_loaded_packages + repaired_plugins
        print(
            f'Updated {", ".join(dict.fromkeys(reasons))}. '
            'Restarting the Colab runtime once to load a consistent NumPy/JAX stack. '
            'After it reconnects, run this setup cell again.',
            flush=True,
        )
        os.kill(os.getpid(), signal.SIGKILL)
else:
    REPO = Path.cwd()
    if not (REPO / 'pyproject.toml').exists():
        REPO = Path.cwd().parents[1]
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'examples' / 'lhc_analysis'))
print(REPO)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/hsbi-toolkit/hnsbi-toolkit


## Generate the configured samples

The shared generator supplies signal, background, and reference samples plus response, resolution, and signal-theory variations. Increase the event counts for publication runs.

In [4]:
from generate_distributions import generate
from hnsbi import Project

EXAMPLE = REPO / 'examples' / 'lhc_analysis'
generate(EXAMPLE / 'data', signal_events=120_000, background_events=300_000, reference_events=400_000)
project = Project.load(EXAMPLE / 'analysis.yaml')
print(project.config.features)


ImportError: cannot import name 'Project' from 'hnsbi' (unknown location)

## Train the reference and native ratio ensembles

Each ratio member has independent class normalization, deterministic train/validation/holdout splits, embedded preprocessing in ONNX, parity checks, and loss/overtraining/calibration/reweighting/normalization diagnostics.

In [ ]:
reference_artifacts = project.train_reference()
reference = reference_artifacts.training.flow
ratio_artifacts = project.train_ratios(reference, normalization_events=40_000, seed=20260729)
for sample, training in ratio_artifacts.training.items():
    print(sample, training.manifest_path)
    print(training.members[0].metadata.get('diagnostics'))


## Systematics, Asimov closure, workspace, and fit

The Asimov weights use sample-wise $E_q[r_k]$ normalization. The workspace is JSON, while the first user interface remains YAML.

In [ ]:
from hnsbi.inference import MinuitInference

systematic_training = project.train_systematics()
runtime_systematics = project.build_runtime_systematics(systematic_training)
asimov = project.build_configured_asimov(reference=reference, ratios=ratio_artifacts.evaluators, normalizer=ratio_artifacts.normalizer, systematics=runtime_systematics)
workspace = project.write_configured_workspace(asimov, reference_manifest=reference_artifacts.checkpoint_manifest, ratio_manifests={name: value.manifest_path for name, value in ratio_artifacts.training.items()})
likelihood = project.workspace_runtime(workspace.path)
fit = MinuitInference(likelihood).fit()
print('raw count / ESS:', asimov.raw_count, asimov.ess)
print(fit.point, fit.errors)
